<style>
.jp-RenderedHTMLCommon h1{color:#ff9900;font-size:2.25em}.jp-RenderedHTMLCommon h2{color:#2563a8}
.jp-RenderedHTMLCommon blockquote{border-left:6px solid #ff9900;background:#fff7e8;padding:.6em 1em}
.jp-RenderedHTMLCommon table{font-size:.9em}.jp-RenderedHTMLCommon code{color:#9a3412}
</style>

# Grok Patterns with AWS Glue
## Parsing web access logs and application events

**Two staged examples · one-line records · bronze-only ingestion**

> Goal: convert loosely structured log lines into named, typed Glue Data Catalog columns.


# Outcomes

This walkthrough establishes how to:

- read Grok syntax from left to right;
- combine built-in patterns with literal separators;
- name captured fields and apply supported numeric casts;
- define reusable custom patterns;
- parse an extended Apache/Iranian e-commerce access log;
- parse a pipe-delimited checkout application log;
- create Grok classifiers and dataset-specific crawlers;
- validate matches, schema, failures, and downstream compatibility.


# Bronze-only target layout

All staged uploads in D341, D342, and D343 remain under the bronze layer.

```text
s3://gksdatalake/bronze/
├── movielens/                       # D341
├── ecommerce/                       # D342
└── logs/                            # D343
    ├── iranian_weblogs/
    │   └── sample-ir-logs.txt
    └── checkout_application/
        └── checkout-events.log
```

Do not upload these raw examples to silver, curated, or serving prefixes. Parsing and cataloging bronze data does not make it curated.


# Why Grok?

Many logs are structured for humans but are not CSV, JSON, or XML:

```text
timestamp | level | service | key=value | message=free text
```

Grok overlays a schema on each line by combining reusable regular-expression patterns. AWS Glue can use a custom Grok classifier during a crawl and write the captured fields as Catalog columns.

Grok is useful when:

- records are one line each;
- separators and field order are predictable;
- parts of a line have recognizable forms;
- named columns are more useful than one raw text column.


# Core Grok syntax

```text
%{PATTERN:field_name}
%{PATTERN:field_name:data_type}
```

Examples:

```grok
%{IPORHOST:client_ip}
%{HTTPDATE:event_time}
%{INT:status_code:int}
%{NUMBER:duration_ms:double}
%{GREEDYDATA:message}
```

- `PATTERN` is a built-in or custom named regular expression.
- `field_name` becomes a Catalog column.
- the optional type cast changes the resulting schema type.
- uncaptured literal text must still match exactly.


# Supported casts and practical typing

AWS Glue Grok fields default to `string`. Supported casts include:

```text
byte, boolean, double, short, int, long, float
```

Examples:

```grok
%{INT:http_status:int}
%{INT:response_bytes:long}
%{NUMBER:amount:double}
```

Dates captured by `HTTPDATE` or `TIMESTAMP_ISO8601` are commonly retained as strings at classification time. Convert them to explicit timestamps in a later transformation with a defined format and timezone policy.

For money, a later decimal cast is usually safer than treating binary floating point as the final business type.


# Built-in pattern vocabulary

Frequently useful names include:

| Pattern | Typical content |
|---|---|
| `IP`, `IPV4`, `IPV6`, `IPORHOST` | network identity |
| `WORD`, `NOTSPACE`, `DATA`, `GREEDYDATA` | general text of increasing breadth |
| `INT`, `NUMBER`, `BASE10NUM` | numeric tokens |
| `HTTPDATE`, `TIMESTAMP_ISO8601` | common timestamp forms |
| `UUID` | request/correlation IDs |
| `URIPATH`, `URIPARAM`, `URIPATHPARAM` | URI components |
| `LOGLEVEL` | common severity names |

Use the narrowest reliable pattern. `GREEDYDATA` is valuable at the end of a record but can hide mistakes when placed early.


# Literal text and escaping

Grok patterns are regular expressions plus named components. Delimiters must be represented.

Input:

```text
2026-09-01T10:15:42Z | INFO | checkout
```

Pattern:

```grok
%{TIMESTAMP_ISO8601:event_time} \| %{LOGLEVEL:level} \| %{WORD:service}
```

The pipe is a regular-expression alternation operator, so `\|` matches a literal pipe. Parentheses, brackets, plus signs, and other regex characters may also require escaping when intended literally.


# Custom patterns

Define a reusable name followed by its regular expression:

```grok
ORDERID O-[0-9]{4,12}
CURRENCY [A-Z]{3}
```

Then reference those names in the main pattern:

```grok
%{ORDERID:order_id}
%{CURRENCY:currency}
```

In the Glue classifier, custom pattern definitions go in **Custom patterns**, one definition per line. The main expression goes in **Grok pattern**.


# Important AWS Glue constraints

- Grok processes **one line at a time**.
- Multiline records and line breaks inside a Grok pattern are not supported.
- The main Grok pattern has a service length limit; keep it focused and reuse custom components.
- A custom classifier is evaluated before built-in classifiers when attached to a crawler.
- Tables created by Glue Grok classifiers use `GrokSerDe`.
- Consumer support for `GrokSerDe` must be checked; AWS specifically advises verifying compatibility with Athena, EMR, and Redshift Spectrum.

Preserve the original bronze log even if a later job writes a more portable columnar representation.


# Example 1 — Iranian e-commerce web logs

Local source inspected:

```text
C:\data\weblogs\sample-ir-logs.txt
```

The full `access.log` is approximately 3.5 GB, so use the smaller sample file first. Its lines follow an **extended Apache combined** layout:

```text
client identity timestamp request status bytes referrer user-agent forwarded-field
```

The final quoted field is an extension beyond the usual combined format. A custom pattern makes this difference explicit.


# Example 1 — sample records

Copy a few representative lines from the local sample into a small test file, or stage the supplied `sample-ir-logs.txt` directly.

```text
5.234.252.242 - - [26/Jan/2019:20:26:21 +0330] "GET /image/64757/productModel/100x100 HTTP/1.1" 200 2402 "https://www.zanbil.ir/browse/cell-phone/" "Mozilla/5.0 (Windows NT 6.1) Chrome/71.0.3578.98 Safari/537.36" "-"
89.36.176.81 - - [26/Jan/2019:20:26:21 +0330] "GET /m/filter/p29%2Ct190?name=mixer HTTP/1.1" 200 19571 "https://example.invalid/referrer" "Mozilla/5.0 (Linux; Android 7.0) Mobile Safari/537.36" "-"
40.77.167.129 - - [22/Jan/2019:03:56:17 +0330] "GET /image/14925/productModel/100x100 HTTP/1.1" 200 1696 "-" "Mozilla/5.0 (compatible; bingbot/2.0)" "-"
```

The shortened URLs/user agents above preserve the record shape. The local source includes percent-encoded Persian URL content and much longer user-agent values.


# Example 1 — parse the line visually

```text
5.234.252.242                  → client_ip
-                              → ident
-                              → authenticated_user
[26/Jan/2019:20:26:21 +0330]  → event_time
"GET ... HTTP/1.1"            → method, request_target, http_version
200                            → status_code
2402                           → response_bytes
"https://..."                  → referrer
"Mozilla/5.0 ..."              → user_agent
"-"                            → forwarded_for
```

Spaces inside quoted referrer and user-agent values mean `NOTSPACE` cannot safely parse those entire fields. Quotes establish their boundaries.


# Example 1 — Grok pattern

Paste this as one line in the classifier's **Grok pattern** field:

```grok
%{IPORHOST:client_ip} %{NOTSPACE:ident} %{NOTSPACE:authenticated_user} \[%{HTTPDATE:event_time}\] "%{WORD:http_method} %{NOTSPACE:request_target} HTTP/%{NUMBER:http_version}" %{INT:status_code:int} (?:%{INT:response_bytes:long}|-) "%{DATA:referrer}" "%{DATA:user_agent}" "%{DATA:forwarded_for}"
```

No custom-pattern block is required for this first version.

The noncapturing group `(?:...|-)` accepts either a numeric byte count or `-`. Because one branch is nonnumeric, validate how the resulting field is represented across the actual sample; a string capture plus later cast may be safer if missing-byte rows exist.


# Example 1 — classifier settings

Create a Grok classifier:

- **Name:** `gks_iranian_weblogs_grok`
- **Type:** Grok
- **Classification:** `iranian_web_access_log`
- **Grok pattern:** the complete one-line expression from the previous slide
- **Custom patterns:** leave empty for the initial version

The classification is a descriptive label written into the Catalog table metadata. It does not transform the log format.

Test the expression against varied lines: successful responses, missing referrers, bots, mobile user agents, query strings, and percent-encoded paths.


# Example 1 — bronze upload

Upload only to:

```text
s3://gksdatalake/bronze/logs/iranian_weblogs/sample-ir-logs.txt
```

For an initial test, a small representative subset is preferable to the multi-gigabyte `access.log`.

Keep the target prefix homogeneous:

- one log format;
- one record per line;
- no ZIP archives mixed with extracted text;
- no CSV lookup files such as hostname or geolocation mappings;
- no notebook, checksum, or temporary files.

Those related datasets require separate bronze prefixes and tables.


# Example 1 — crawler and validation

Suggested resources:

- database: `gks_logs_bronze`
- crawler: `gks_iranian_weblogs_crawler`
- source: `s3://gksdatalake/bronze/logs/iranian_weblogs/`
- classifier: `gks_iranian_weblogs_grok` first in the custom-classifier list

After the crawl, verify table location, classification, and columns. Inspect especially:

- status and byte columns are numeric where expected;
- full request targets remain intact;
- referrer/user-agent values do not shift columns;
- timezone text remains part of `event_time`;
- unmatched or malformed lines are identified through logs and downstream checks.


# Example 2 — checkout application events

This format demonstrates capabilities not emphasized by the web log:

- literal pipe separators;
- ISO-8601 timestamps;
- UUID request IDs;
- reusable custom business patterns;
- typed amount and latency;
- a free-text message at the end.

Target:

```text
s3://gksdatalake/bronze/logs/checkout_application/checkout-events.log
```

One event occupies exactly one physical line.


# Example 2 — stage `checkout-events.log`

Copy the block into a plain-text editor and save as **`checkout-events.log`** with UTF-8 encoding.

```text
2026-09-01T10:15:42.184Z | INFO | checkout | req=550e8400-e29b-41d4-a716-446655440000 | order=O-1001 | amount=1898.00 INR | latency_ms=142 | message=Payment authorized
2026-09-01T10:16:03.027Z | WARN | checkout | req=6ba7b810-9dad-41d1-80b4-00c04fd430c8 | order=O-1002 | amount=747.00 INR | latency_ms=816 | message=Inventory confirmation slow
2026-09-01T10:17:55.901Z | ERROR | checkout | req=6ba7b811-9dad-41d1-80b4-00c04fd430c8 | order=O-1003 | amount=1299.50 INR | latency_ms=95 | message=Payment declined by issuer
2026-09-01T10:18:12.330Z | INFO | refund | req=6ba7b812-9dad-41d1-80b4-00c04fd430c8 | order=O-1001 | amount=499.50 INR | latency_ms=211 | message=Partial refund submitted
```

Do not wrap long lines in the saved file. Visual wrapping in an editor is acceptable; physical newline insertion is not.


# Example 2 — custom patterns

Paste these into **Custom patterns**, one definition per line:

```grok
ORDERID O-[0-9]{4,12}
CURRENCY [A-Z]{3}
SERVICENAME [a-z][a-z0-9_-]*
```

Meaning:

- `ORDERID` enforces `O-` followed by 4–12 digits;
- `CURRENCY` accepts a three-letter uppercase code;
- `SERVICENAME` allows lowercase service identifiers with digits, `_`, or `-`.

These names are reusable building blocks, not output columns until referenced with a field name.


# Example 2 — Grok pattern

Paste as one physical line:

```grok
%{TIMESTAMP_ISO8601:event_time} \| %{LOGLEVEL:log_level} \| %{SERVICENAME:service} \| req=%{UUID:request_id} \| order=%{ORDERID:order_id} \| amount=%{NUMBER:amount:double} %{CURRENCY:currency} \| latency_ms=%{INT:latency_ms:long} \| message=%{GREEDYDATA:message}
```

Expected columns:

```text
event_time, log_level, service, request_id, order_id,
amount, currency, latency_ms, message
```

`GREEDYDATA` is last, so it safely consumes the remainder without stealing delimiters needed by later fields.


# Example 2 — classifier and crawler

Classifier:

- **Name:** `gks_checkout_events_grok`
- **Type:** Grok
- **Classification:** `checkout_application_log`
- **Grok pattern:** the complete expression
- **Custom patterns:** the three definitions

Crawler:

- **Name:** `gks_checkout_events_crawler`
- **Source:** `s3://gksdatalake/bronze/logs/checkout_application/`
- **Database:** `gks_logs_bronze`
- **Classifier order:** `gks_checkout_events_grok` first
- **Schedule:** on demand for staged validation

Use a crawler IAM role scoped to this bronze prefix.


# Example 2 — validate types and semantics

Confirm:

| Field | Expected interpretation |
|---|---|
| `event_time` | captured timestamp text; parse explicitly later |
| `log_level` | INFO/WARN/ERROR string |
| `service` | checkout/refund string |
| `request_id` | UUID string |
| `order_id` | custom-pattern string |
| `amount` | double at crawl schema; convert to decimal later |
| `currency` | three-letter string |
| `latency_ms` | long integer |
| `message` | remaining free text |

Also verify that all four lines match and no entire line falls into a single fallback column.


# Pattern-building method

Build from left to right:

1. Start with the first stable token.
2. Add the exact delimiter.
3. Capture the next token.
4. Test several representative lines.
5. Continue until the entire line is consumed.
6. Replace broad patterns with narrower ones where useful.
7. Add type casts only after matching is stable.
8. Test exceptions: missing values, long URLs, bots, Unicode, and failure messages.

An online Grok debugger may help, but AWS warns that debugger behavior can differ from Glue. The crawler result remains the authoritative validation for this implementation.


# Anchoring and full-line matching

When accidental partial matches are possible, consider explicit anchors:

```grok
^%{TIMESTAMP_ISO8601:event_time} ... %{GREEDYDATA:message}$
```

- `^` means start of line.
- `$` means end of line.

Anchors make the format contract clearer, but they also reject lines containing unexpected leading/trailing text. Decide whether strict rejection or permissive capture is appropriate for bronze ingestion, then measure unmatched records rather than silently losing them.


# Optional fields require careful design

Example source variation:

```text
... bytes=2402 ...
... bytes=- ...
```

Possible strategies:

- capture as string and normalize `-` to null later;
- use a noncapturing alternative such as `(?:%{INT:bytes}|-)`;
- define a custom pattern that accepts both forms;
- split materially different formats into separate prefixes/classifiers.

A field-level type cast and an alternate nonnumeric token may conflict. Prefer reliable matching in bronze, then apply strict typing with explicit error handling downstream.


# Troubleshooting map

| Symptom | Likely cause | First correction |
|---|---|---|
| Classification is `UNKNOWN` | classifier did not match | compare exact delimiters and line endings |
| One field consumes too much | `GREEDYDATA` used too early | replace with `DATA`, `NOTSPACE`, or a custom pattern |
| Columns shift on web logs | quoted fields parsed as spaces | preserve quote boundaries in pattern |
| Pipe pattern behaves strangely | pipe not escaped | use `\|` for a literal pipe |
| Some lines disappear/fail | format variants or multiline records | sample unmatched lines and separate variants |
| Numeric cast fails | `-`, empty, or mixed values | capture as string, normalize later |
| Custom name not found | missing definition or typo/case | verify one-definition-per-line block |
| Correct test, wrong table | old crawler state/classifier | use a new crawler/test table for correction |


# Grok classifier operational notes

- Custom classifiers run before built-in classifiers in their configured order.
- Changing a classifier does not guarantee reclassification of data already remembered by a crawler.
- For a corrected pattern, create a new crawler targeting a test database/table and compare results.
- Keep one log format per source prefix where possible.
- Use exclusions for archives, lookup CSVs, temporary files, and malformed quarantine files.
- Select schema-change behavior intentionally before recurring crawls.
- Retain raw bronze data so parsing logic can be improved and replayed.


# Downstream design

Grok-classified bronze tables expose structure, but a later processing step should typically:

- preserve the original line for traceability where required;
- parse timestamps with timezone awareness;
- turn `-` and empty tokens into nulls;
- cast money to fixed-precision decimal;
- normalize HTTP method/status and service/log-level values;
- URL-decode request targets only when appropriate and safe;
- separate accepted and rejected records;
- write efficient columnar data to a non-bronze layer.

D343 performs only bronze staging, classification, crawling, and validation.


# Bronze-path audit across the sequence

| Notebook | Dataset | Upload prefix |
|---|---|---|
| D341 | MovieLens movies/ratings | `s3://gksdatalake/bronze/movielens/` |
| D342 | JSON orders/XML invoices | `s3://gksdatalake/bronze/ecommerce/` |
| D343 | Iranian/application logs | `s3://gksdatalake/bronze/logs/` |

All raw uploads are restricted to bronze. Any future transformation into silver/curated data belongs in a separate workflow and notebook.


# Final checklist

- [ ] One physical line per Grok record
- [ ] Representative sample includes normal and exceptional values
- [ ] Exact spaces, quotes, brackets, pipes, and literals represented
- [ ] Built-in patterns are narrow enough
- [ ] Custom patterns are defined one per line and referenced correctly
- [ ] Numeric casts tolerate actual source values
- [ ] Classifier is first in the crawler's custom-classifier order
- [ ] Upload target is under `s3://gksdatalake/bronze/logs/`
- [ ] Source prefix contains only the intended format
- [ ] Catalog columns and classification verified
- [ ] Unmatched records and consumer `GrokSerDe` support assessed


# Summary

```text
One log line
   ↓
Grok expression = built-in patterns + literals + custom patterns
   ↓
Named captures and optional numeric casts
   ↓
Glue Grok classifier
   ↓
Crawler writes bronze table metadata
   ↓
Validate matches before downstream transformation
```

Two capabilities were demonstrated:

1. Parsing a complex quoted Apache-style record with URLs and user agents.
2. Parsing a custom business log with delimiters, UUIDs, typed metrics, and reusable patterns.

## References

- [AWS Glue: writing custom Grok classifiers](https://docs.aws.amazon.com/glue/latest/dg/custom-classifier.html)
- [AWS Glue: creating classifiers in the console](https://docs.aws.amazon.com/glue/latest/dg/console-classifiers.html)
- [AWS Glue: Grok classifier API fields](https://docs.aws.amazon.com/glue/latest/webapi/API_GrokClassifier.html)
